[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/sh2026-workshop/workshop/full_day/04_linking_and_ambiguity.ipynb)

# 04 · Linking, ambiguity and historical geography

**Spatial Humanities 2026 workshop**

**Official workshop title:** *AI and NLP for Spatial Humanities: From Manual Annotation to LLM-Assisted Interpretation*

This notebook moves from **recognition** ("this string looks like a place") to **resolution** ("which place does it refer to?").

## Learning goals
By the end, you should be able to:
- distinguish named-entity recognition from entity linking/geocoding;
- inspect candidate ambiguity instead of hiding it;
- understand why a coordinate is an interpretive decision;
- identify cases where present-day gazetteers are historically inappropriate;
- preserve source forms and route uncertain resolutions to human review.

> **Key message:** A coordinate is an interpretation, not simply an annotation.

In [ ]:
!wget -q https://raw.githubusercontent.com/IgnatiusEzeani/spatial-humanities-2026/sh2026-workshop/workshop/sh2026_setup.py

import sh2026_setup as sh
ctx = sh.setup()

from workshop_support.display import (show_spans, compare_spans, show_journey,
                                      show_fields, checkpoint)


## 1. Recognition is not resolution

Suppose an NER model marks **Cambridge** as a place. That is only the first question.

Resolution asks:
- Cambridge, England?
- Cambridge, Massachusetts?
- another Cambridge?

A gazetteer typically returns one or more candidate records. Choosing one candidate may require document context, historical knowledge, or a human decision.

In [ ]:
from spatio_textual.geocode import GeoResolver

resolver = GeoResolver(max_candidates=5)

examples = ["London", "Cambridge", "Amsterdam", "Czechoslovakia", "Atlantis"]
rows = []
for name in examples:
    result = resolver.resolve(name, label="GPE", context=f"The text mentions {name}.")
    rows.append({"source_text": name, **(result or {})})


**Inspect the output carefully.** Useful audit fields include:

- `resolved_name`
- `resolution_status`
- `geo_source`
- `geo_confidence`
- `ambiguous`
- `candidates_count`
- `candidates`

The important question is not merely *"did we get coordinates?"* but *"what evidence justified these coordinates?"*

In [ ]:
# A compact table for inspection.
import pandas as pd

cols = ["source_text", "resolved_name", "resolution_status", "place_type_resolved", "lat", "lon", "geo_confidence", "ambiguous", "candidates_count"]
df = pd.DataFrame(rows)
for col in cols:
    if col not in df.columns:
        df[col] = None
display(df[cols].fillna(""))

## 2. Ambiguity as a first-class result

If a name has several plausible candidates, `GeoResolver` marks the result as ambiguous rather than pretending that ranking equals certainty.

Try changing the preferred country below. `prefer_country` is a **ranking preference**, not evidence that the preferred candidate is historically correct.

In [ ]:
preference_rows = []
for preference in [None, "GB", "US"]:
    r = GeoResolver(prefer_country=preference, max_candidates=5)
    out = r.resolve("Cambridge", label="GPE", context="I travelled from Cambridge to London.") or {}
    preference_rows.append({
        "ranking_preference": preference or "none",
        "resolved_name": out.get("resolved_name"),
        "country": out.get("country_code") or out.get("countrycode"),
        "lat": out.get("lat"),
        "lon": out.get("lon"),
        "ambiguous": out.get("ambiguous"),
        "candidates": out.get("candidates_count"),
    })

display(pd.DataFrame(preference_rows).fillna(""))
checkpoint("A ranking preference changes ordering; it does not supply historical evidence.")


### Discussion

1. Should population be the default ranking heuristic?
2. What evidence in the surrounding text could disambiguate a place?
3. When should the system refuse to decide?
4. Should a human correction overwrite the machine suggestion or be stored alongside it?

For this project, the preferred design is **append-only provenance**: keep the model suggestion and record the human decision separately.

## 3. Historical geography: why "unresolved" can be the better answer

Present-day gazetteers are not historical GIS systems. A historical polity should not be silently collapsed onto a current state simply because a modern database requires one coordinate.

The SH2026 branch therefore preserves **Czechoslovakia** as the source string and marks it as a historical polity requiring time-aware or human resolution.

In [ ]:
historical = resolver.resolve(
    "Czechoslovakia",
    label="GPE",
    context="The narrator described a journey through Czechoslovakia before later border changes.",
)

show_fields(historical, [
    "resolved_name", "place_type_resolved", "resolution_status", "lat", "lon", "review_notes"
])

expected = {
    "resolved_name": "Czechoslovakia",
    "place_type_resolved": "HISTORICAL_POLITY",
    "resolution_status": "unresolved",
}
for field, value in expected.items():
    assert historical.get(field) == value, f"Expected {field}={value!r}; got {historical.get(field)!r}"
assert historical.get("lat") is None and historical.get("lon") is None, "Historical polity should remain unlocated"
checkpoint("The historical name is preserved without a misleading modern coordinate.")


This gives us an important methodological rule:

> **Preserve the historical source form before attempting modern normalization.**

A time-aware gazetteer could later attach period-specific geometries, but that is a different task from ordinary present-day geocoding.

## 4. A small human-review record

Rather than replacing the machine output, create a review record that preserves:
- source string;
- machine suggestion;
- reviewer decision;
- reason;
- timestamp/version if available.

In [ ]:
from datetime import datetime, timezone

machine = resolver.resolve("Cambridge", label="GPE", context="I left Cambridge and travelled to London.")

review = {
    "source_text": "Cambridge",
    "context": "I left Cambridge and travelled to London.",
    "machine_resolution": machine,
    "human_decision": {
        "status": "accepted_for_teaching_example",
        "resolved_name": machine.get("resolved_name") if machine else None,
        "reason": "Illustrative review only; real research requires document-level contextual evidence.",
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
    }
}

print(json.dumps(review, indent=2, ensure_ascii=False))

## 5. Resolution confidence is not historical truth

A numerical confidence score can summarize the resolver's own evidence, but it does **not** mean:
- 95% probability that a historical interpretation is correct;
- 95% agreement among scholars;
- 95% certainty that the narrator intended that place.

Confidence is meaningful only when its provenance and calculation are understood.

## 6. Exercise: decide what should be mapped

For each phrase, decide whether you would:

1. map directly;
2. map with an ambiguity flag;
3. preserve without coordinates;
4. request human/historical review.

- "London"
- "Cambridge"
- "Czechoslovakia"
- "the village"
- "beyond the river"
- "home"

Notice that several spatially meaningful expressions are **not failures** simply because they cannot be represented as one point on a modern basemap.

### Exercise: what would you map?

Take two minutes with the person next to you. For each expression, choose one:

**1** map it · **2** map it with an ambiguity flag · **3** retain it without
coordinates · **4** send it to human review

| Expression | Expression |
|---|---|
| London | Cambridge |
| Czechoslovakia | the village |
| beyond the river | home |

We will take a show of hands on `the village` and `Czechoslovakia`. Several are
spatially meaningful but have no defensible modern point. That is not a failure
of the method.


## 7. Take-away

We now have four distinct layers:

**Recognition → Resolution → Relation → Interpretation**

A system may perform very well at recognition while still being uncertain at resolution. Historical and experiential spatial references can also be meaningful without a defensible coordinate.

**Next:** affect and narrator-centred events, where the same audit principle becomes even more important: computational labels are signals for analysis, not psychological ground truth.